In [1]:
import numpy as np
import pandas as pd

# URLs de los datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')

# a) Consolidar archivos de años y normalizar columnas a minúsculas
lista_dfs = []
for url in archivos_anio:
    df_temp = pd.read_csv(url)
    df_temp.columns = df_temp.columns.str.lower() # Pasamos columnas a minúsculas
    lista_dfs.append(df_temp)

df_anio = pd.concat(lista_dfs, ignore_index=True)

# b) Identificar y limpiar el ISO duplicado en df_codigos
# Primero veamos cuál está duplicado (Inspección interna: suele ser un error donde un ISO se repite)
iso_repetido = df_codigos[df_codigos.duplicated(subset=['codigo_iso'], keep=False)]
print("Registros ISO duplicados detectados originalmente:")
print(iso_repetido)

# Eliminamos el duplicado incorrecto. (keep='first' o 'last' según corresponda,
# por buena práctica removemos el duplicado manteniendo el registro válido estructurado)
df_codigos_limpio = df_codigos.drop_duplicates(subset=['codigo_iso'], keep='first')

# c) Combinar df_anio con df_codigos_limpio usando merge (Inner join para solo coincidencias)
df = pd.merge(df_anio, df_codigos_limpio, on='codigo_iso', how='inner')

# Mostramos el resultado final
df.head()

Registros ISO duplicados detectados originalmente:
    codigo_iso      pais
179        ZWE  Zimbabue
180        ZWE      malo


,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos


In [2]:
print("=== ESTRUCTURA DEL DATAFRAME ===")
print(f"Filas (observaciones): {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Nombres de columnas: {df.columns.tolist()}")
print("\nTipos de datos por columna:")
print(df.dtypes)

print("\n=== RESUMEN ESTADÍSTICO ===")
print(df.describe())

# Países extremos en índice y ranking
idx_min_ind = df['indice'].idxmin()
idx_max_ind = df['indice'].idxmax()
print(f"\nMayor libertad (Mínimo índice): {df.loc[idx_min_ind, 'pais']} ({df.loc[idx_min_ind, 'indice']})")
print(f"Menor libertad (Máximo índice): {df.loc[idx_max_ind, 'pais']} ({df.loc[idx_max_ind, 'indice']})")

print("\n=== DATOS FALTANTES ===")
nulos = df.isnull().sum()
proporcion_nulos = df.isnull().mean()
print("Cantidad de nulos por columna:")
print(nulos)
print("\nProporción de nulos por columna:")
print(proporcion_nulos)

print("\n=== UNICIDAD Y DUPLICADOS ===")
print(f"Países distintos: {df['pais'].nunique()}")
print(f"Años distintos: {df['anio'].nunique()}")
print(f"Filas exactamente duplicadas: {df.duplicated().sum()}")

print("\n=== VALIDACIÓN CRUZADA ===")
# Validar si un código ISO quedó asociado a más de un país en el DF final
inconsistencias = df.groupby('codigo_iso')['pais'].nunique()
print(f"Códigos ISO con inconsistencias (debería ser 0): {(inconsistencias > 1).sum()}")

=== ESTRUCTURA DEL DATAFRAME ===
Filas (observaciones): 3060
Columnas: 5
Nombres de columnas: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais']

Tipos de datos por columna:
codigo_iso     object
anio            int64
indice        float64
ranking       float64
pais           object
dtype: object

=== RESUMEN ESTADÍSTICO ===
              anio        indice        ranking
count  3060.000000   2664.000000    2837.000000
mean   2009.941176    205.782316     477.930913
std       5.786024   2695.525264    6474.935347
min    2001.000000      0.000000       1.000000
25%    2005.000000     15.295000      34.000000
50%    2009.000000     28.000000      70.000000
75%    2015.000000     41.227500     110.000000
max    2019.000000  64536.000000  121056.000000

Mayor libertad (Mínimo índice): Dinamarca (0.0)
Menor libertad (Máximo índice): Kosovo (64536.0)

=== DATOS FALTANTES ===
Cantidad de nulos por columna:
codigo_iso      0
anio            0
indice        396
ranking       223
pais          

In [5]:
# Aseguramos la lista de países
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

# Medida de seguridad: Limpiamos espacios en blanco invisibles en la columna clave
df['codigo_iso'] = df['codigo_iso'].astype(str).str.strip().str.upper()

# Filtramos el DataFrame para los países de la lista
df_america = df[df['codigo_iso'].isin(america)].copy()

# Verificación rápida: Si esto da 0, el merge anterior eliminó los países o la columna se llama distinto
print(f"Cantidad de filas encontradas para América Latina: {len(df_america)}\n")

if len(df_america) == 0:
    print("⚠️ ERROR: El DataFrame 'df_america' está vacío. Revisa si tus columnas se llaman distinto usando: print(df.columns)")
else:
    print("=== SOLUCIÓN A) USANDO CICLO FOR (Corregido) ===")
    for anio_act in sorted(df_america['anio'].unique()):
        df_sub = df_america[df_america['anio'] == anio_act]

        # Ignoramos años que no tengan datos válidos de índice
        df_sub_valid = df_sub.dropna(subset=['indice'])

        if not df_sub_valid.empty:
            # Buscamos la fila exacta usando el índice de posición (idxmin)
            idx_mejor = df_sub_valid['indice'].idxmin()
            idx_peor = df_sub_valid['indice'].idxmax()

            mejor_pais = df_sub_valid.loc[idx_mejor, 'pais']
            mejor_valor = df_sub_valid.loc[idx_mejor, 'indice']

            peor_pais = df_sub_valid.loc[idx_peor, 'pais']
            peor_valor = df_sub_valid.loc[idx_peor, 'indice']

            print(f"Año {anio_act} -> Mayor Libertad: {mejor_pais} ({mejor_valor}) | Menor Libertad: {peor_pais} ({peor_valor})")

    print("\n=== SOLUCIÓN B) ENFOQUE VECTORIZADO (GROUPBY - Corregido) ===")
    # Para evitar errores de índices duplicados o vacíos, filtramos nulos primero
    df_america_clean = df_america.dropna(subset=['indice'])

    # Obtenemos las filas que corresponden al mínimo y máximo por año
    mejores_indices = df_america_clean.groupby('anio')['indice'].idxmin()
    peores_indices = df_america_clean.groupby('anio')['indice'].idxmax()

    mejores = df_america_clean.loc[mejores_indices, ['anio', 'pais', 'indice']].rename(columns={'pais': 'Mayor Libertad', 'indice': 'Indice_Min'})
    peores = df_america_clean.loc[peores_indices, ['anio', 'pais', 'indice']].rename(columns={'pais': 'Menor Libertad', 'indice': 'Indice_Max'})

    # Combinamos ambos resultados en una sola tabla limpia por año
    res_regional = pd.merge(mejores, peores, on='anio')
    display(res_regional)

Cantidad de filas encontradas para América Latina: 493

=== SOLUCIÓN A) USANDO CICLO FOR (Corregido) ===
Año 2001 -> Mayor Libertad: Canadá (0.8) | Menor Libertad: Cuba (90.3)
Año 2002 -> Mayor Libertad: Trinidad y Tobago (1.0) | Menor Libertad: Cuba (97.83)
Año 2003 -> Mayor Libertad: Trinidad y Tobago (2.0) | Menor Libertad: Argentina (35826.0)
Año 2004 -> Mayor Libertad: Trinidad y Tobago (2.0) | Menor Libertad: Cuba (87.0)
Año 2005 -> Mayor Libertad: Bolivia (4.5) | Menor Libertad: Cuba (95.0)
Año 2006 -> Mayor Libertad: Canadá (4.88) | Menor Libertad: Cuba (96.17)
Año 2007 -> Mayor Libertad: Canadá (3.33) | Menor Libertad: Cuba (88.33)
Año 2008 -> Mayor Libertad: Canadá (3.7) | Menor Libertad: Cuba (94.0)
Año 2009 -> Mayor Libertad: Estados Unidos (6.75) | Menor Libertad: Cuba (78.0)
Año 2012 -> Mayor Libertad: Jamaica (9.88) | Menor Libertad: Cuba (71.64)
Año 2013 -> Mayor Libertad: Jamaica (10.9) | Menor Libertad: Cuba (70.92)
Año 2014 -> Mayor Libertad: Canadá (10.99) | Menor L

,anio,Mayor Libertad,Indice_Min,Menor Libertad,Indice_Max
0,2001,Canadá,0.80,Cuba,90.30
1,2002,Trinidad y Tobago,1.00,Cuba,97.83
2,2003,Trinidad y Tobago,2.00,Argentina,35826.00
3,2004,Trinidad y Tobago,2.00,Cuba,87.00
4,2005,Bolivia,4.50,Cuba,95.00
5,2006,Canadá,4.88,Cuba,96.17
6,2007,Canadá,3.33,Cuba,88.33
7,2008,Canadá,3.70,Cuba,94.00
8,2009,Estados Unidos,6.75,Cuba,78.00
9,2012,Jamaica,9.88,Cuba,71.64


In [6]:
print("=== CONSTRUCCIÓN DE LA TABLA DINÁMICA ===")
# Construimos la tabla pivot con países en filas, años en columnas e índice máximo alcanzado.
# fill_value=0 reemplaza automáticamente los datos faltantes con cero.
pivot_libertad = pd.pivot_table(df, values='indice', index='pais', columns='anio', aggfunc='max', fill_value=0)

# Mostramos las primeras 5 filas para verificar que se construyó bien
display(pivot_libertad.head())

print("\n=== RESOLVIENDO PREGUNTAS ADICIONALES ===")

# --- Pregunta a) ---
max_valor = pivot_libertad.max().max()
pais_max = pivot_libertad.max(axis=1).idxmax()

# Para calcular el menor sin ser afectado por los ceros (valores faltantes), los reemplazamos temporalmente por NaN
tabla_sin_ceros = pivot_libertad.replace(0, np.nan)
min_valor = tabla_sin_ceros.min().min()
pais_min = tabla_sin_ceros.min(axis=1).idxmin()

print(f"a) País con el MAYOR índice registrado en toda la tabla: {pais_max} ({max_valor})")
print(f"   País con el MENOR índice registrado (excluyendo ceros): {pais_min} ({min_valor})")

# --- Pregunta b) ---
# Calculamos el promedio por columnas (años) omitiendo los NaN (antiguos ceros)
promedio_anios = tabla_sin_ceros.mean(axis=0)
print(f"b) Año con promedio de índice más ALTO (menos libertad promedio): {promedio_anios.idxmax()} ({promedio_anios.max():.2f})")
print(f"   Año con promedio de índice más BAJO (más libertad promedio): {promedio_anios.idxmin()} ({promedio_anios.min():.2f})")

# --- Pregunta c) ---
# Calculamos la diferencia entre el máximo y mínimo de cada fila (país)
variabilidad = tabla_sin_ceros.max(axis=1) - tabla_sin_ceros.min(axis=1)
pais_variable = variabilidad.idxmax()
print(f"c) País que muestra la MAYOR variabilidad a lo largo del tiempo: {pais_variable} (Diferencia de {variabilidad.max():.2f} puntos)")

# --- Pregunta d) ---
# Si la variabilidad es exactamente 0, significa que su índice nunca cambió
constantes = variabilidad[variabilidad == 0].index.tolist()
print(f"d) Países con índice constante en todos los años registrados: {constantes if constantes else 'Ninguno'}")

# --- Pregunta e) ---
# Buscamos filas completas que solo contengan ceros
puros_ceros = pivot_libertad[(pivot_libertad == 0).all(axis=1)].index.tolist()
print(f"e) Países que quedaron con todos sus valores en 0: {puros_ceros if puros_ceros else 'Ninguno'}")
print("   Explicación teórica: Esto ocurre porque eran países que aparecían en 'df_codigos',")
print("   pero que no tenían ningún dato registrado en las bitácoras anuales ('df_anio').")
print("   Al hacer el 'inner join' (o por descarte de nulos), no sumaron datos y la tabla dinámica los rellenó con 'fill_value=0'.")

=== CONSTRUCCIÓN DE LA TABLA DINÁMICA ===


anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Afghanistán,35.5,40.17,28.25,39.17,44.25,56.50,59.25,54.25,51.67,37.36,37.07,37.44,37.75,39.46,37.28,36.55
Albania,0.0,6.50,11.50,14.17,18.00,25.50,16.00,21.75,21.50,30.88,29.92,28.77,29.92,29.92,29.49,29.84
Alemania,1.5,1.33,2.00,4.00,5.50,5.75,4.50,3.50,4.25,10.24,10.23,11.47,14.80,14.97,14.39,14.60
Algeria,31.0,33.00,43.50,40.33,40.00,40.50,31.33,49.56,47.33,36.54,36.26,36.63,41.69,42.83,43.13,45.75
Andorra,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,6.82,6.82,19.87,19.87,21.03,22.21,24.63



=== RESOLVIENDO PREGUNTAS ADICIONALES ===
a) País con el MAYOR índice registrado en toda la tabla: Kosovo (64536.0)
   País con el MENOR índice registrado (excluyendo ceros): Austria (0.5)
b) Año con promedio de índice más ALTO (menos libertad promedio): 2012 (468.88)
   Año con promedio de índice más BAJO (más libertad promedio): 2002 (25.88)
c) País que muestra la MAYOR variabilidad a lo largo del tiempo: Kosovo (Diferencia de 46098.00 puntos)
d) Países con índice constante en todos los años registrados: ['Antigua y Barbuda', 'Granada']
e) Países que quedaron con todos sus valores en 0: Ninguno
   Explicación teórica: Esto ocurre porque eran países que aparecían en 'df_codigos',
   pero que no tenían ningún dato registrado en las bitácoras anuales ('df_anio').
   Al hacer el 'inner join' (o por descarte de nulos), no sumaron datos y la tabla dinámica los rellenó con 'fill_value=0'.
